PHASE 6: ML Models

Trains 6 puprose-built models: 
1. Stint Length Predictor (Gradient Boosting Regressor)
2. Compound Selector (Gradient Boosting Classifier)
3. Strategy Outcome Predictor (Gradient Boosting Classifier)
4. Pit Window Predictor (Gradient Boosting Regressor)
5. Circuit Clustering (K-Means)
6. Undercut Viability Classifier (Logistic Regression)

In [2]:
import pandas as pd
import numpy as np
import os
import logging
import warnings
import joblib

from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score

warnings.filterwarnings('ignore')

SETUP

In [3]:
BASE = r'C:\Users\adity\Desktop\BoxBox'

os.makedirs(os.path.join(BASE, 'models'), exist_ok=True)
os.makedirs(os.path.join(BASE, 'data', 'outputs'), exist_ok=True)

logger_name = __name__  
log = logging.getLogger(logger_name)
if log.hasHandlers():
    log.handlers.clear()

log.setLevel(logging.INFO)

formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

file_handler = logging.FileHandler(
    r'C:\Users\adity\Desktop\BoxBox\data\outputs\phase1_log.txt'
)
file_handler.setFormatter(formatter)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)

log.addHandler(file_handler)
log.addHandler(stream_handler)

COMPOUND_MAP = {'SOFT': 2, 'MEDIUM': 1, 'HARD': 0}

LOAD ALL THE REQUIRED DATA

In [4]:
def load_data():
    log.info("Loading data...")

    engineered = pd.read_csv(
        os.path.join(BASE, 'data', 'processed', 'engineered_laps.csv')
    )
    feature_store = pd.read_csv(
        os.path.join(BASE, 'data', 'processed', 'feature_store.csv')
    )
    pit_stops = pd.read_csv(
        os.path.join(BASE, 'data', 'processed', 'pit_stops_clean.csv')
    )
    race_results = pd.read_csv(
        os.path.join(BASE, 'data', 'raw', 'race_results.csv')
    )
    raw_laps = pd.read_csv(
        os.path.join(BASE, 'data', 'raw', 'raw_laps.csv')
    )

    log.info(f" Engineered_laps: {len(engineered)} rows")
    log.info(f" feature_stops: {len(feature_store)} rows")
    log.info(f" pit_stops_clean: {len(pit_stops)} rows")
    log.info(f" race_results: {len(race_results)} rows")
    log.info(f" raw_laps: {len(raw_laps)} rows")

    return engineered, feature_store, pit_stops, race_results, raw_laps

MODEL 1 - STINT LENGTH PREDICTOR

Predicts realistic stint length given compound, circuit type, and conditions. Trained on actual 2024 stint lengths.

In [5]:
def train_stint_length_predictor(engineered):
    log.info("MODEL-1: STINT LENGTH PREDICTOR")
    log.info("-" * 50)

    stint_data = engineered.groupby(
        ['CircuitName', 'Driver', 'Stint', 'Compound']
    ).agg(
        StintLength = ('TyreLife', 'max'),
        AvgTrackTemp = ('TrackTemp', 'mean'),
        CircuitTypeEncoded = ('CircuitTypeEncoded', 'first'),
        CompoundEncoded = ('CompoundEncoded', 'first'),
        SafetyCarProbability = ('SafetyCarProbability', 'first')
    ).reset_index().dropna()

    features = ['CompoundEncoded', 'CircuitTypeEncoded',
                'AvgTrackTemp', 'SafetyCarProbability']
    X = stint_data[features]
    y = stint_data['StintLength']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = GradientBoostingRegressor(
        n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)

    log.info(f" Trained on {len(X_train)} stints, tested on {len(X_test)}")
    log.info(f" MAE: {mae:.2f} laps")

    return model, {'model': 'StintLengthPredictor', 'MAE_laps': round(mae, 2)}

MODEL 2: COMPOUND SELECTOR

Multi-class classifier predicting which compound is used given circuit type, temperature, and stage of the race

In [6]:
def train_compound_selector(engineered):
    log.info("MODEL 2: COMPOUND SELECTOR")
    log.info("-" * 50)

    stint_data = engineered.groupby(
        ['CircuitName', 'Driver', 'Stint']
    ).agg(
        Compound = ('Compound', 'first'),
        AvgTrackTemp = ('TrackTemp', 'mean'),
        CircuitTypeEncoded = ('CircuitTypeEncoded', 'first'),
        StintNumber = ('Stint', 'first'),
        SafetyCarProbability = ('SafetyCarProbability', 'first')
    ).reset_index(drop=True).dropna()

    features = ['CircuitTypeEncoded', 'AvgTrackTemp',
                'StintNumber', 'SafetyCarProbability']

    X = stint_data[features]
    y = stint_data['Compound']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model = GradientBoostingClassifier(
        n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='weighted')

    log.info(f" Trained on {len(X_train)} stints, tested on {len(X_test)}")
    log.info(f" Accuracy: {acc:.3f} | Weighted F1: {f1:.3f}")

    return model, {'model': 'CompoundSelector', 'Accuracy': round(acc,3),
                   'F1': round(f1, 3)}

MODEL 3 - STRATEGY OUTCOME PREDICTOR

Predicts position gain / neutral / loss given grid position, number of stops, and circuit characterstics

In [7]:
def train_strategy_outcome_predictor(engineered, race_results, feature_store):
    log.info("MODEL 3 - STRATEGY OUTCOME PREDICTOR")
    log.info("-" * 50)

    #Number of stops per driver per circuit (max stint - 1)
    stops = engineered.groupby(['CircuitName', 'Driver'])['Stint'].max().reset_index()
    stops.columns = ['CircuitName', 'Driver', 'MaxStint']
    stops['NumStops'] = stops['MaxStint'] - 1

    #Merge with race results - race_results uses 'Abbreviation' as driver code
    results = race_results.rename(columns = {'Abbreviation': 'Driver'})
    merged = stops.merge(
        results[['CircuitName', 'Driver', 'GridPosition', 'PositionChange']],
        on = ['CircuitName', 'Driver'], how = 'inner'
    )

    merged = merged.merge(
        feature_store[['CircuitName', 'CircuitTypeEncoded', 'SafetyCarProbability']],
        on = 'CircuitName', how='left'
    )

    merged = merged.dropna(subset = ['GridPosition', 'PositionChange'])

    # Bucketing outcome into 3 classes
    def bucket_outcome(change):
        if change >= 2:
            return 'Gain'
        elif change <= 2:
            return 'Loss'
        else:
            return 'Neutral'

    merged['Outcome'] = merged['PositionChange'].apply(bucket_outcome)

    features = ['GridPosition', 'NumStops', 'CircuitTypeEncoded', 'SafetyCarProbability']
    X = merged[features]
    y = merged['Outcome']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model = GradientBoostingClassifier(
        n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='weighted')

    log.info(f" Trained on {len(X_train)} driver-races, tested on {len(X_test)}")
    log.info(f" Accuracy: {acc:.3f} | Weighted F1: {f1:.3f}")
    log.info(f" Outcome distribution:\n{merged['Outcome'].value_counts().to_string()}")

    return model, {'model': 'StrategyOutcomePredictor', 'Accuracy': round(acc, 3),
                   'F1': round(f1, 3)}

MODEL 4 - PIT WINDOW PREDICTOR

Predicts the lap on which a pit stop is likely to happen, given stint number, outgoing compound, and circuit

In [8]:
def train_pit_window_predictor(pit_stops, feature_store):
    log.info("MODEL 4 - PIT WINDOW PREDICTOR")
    log.info("-" * 50)

    data = pit_stops.merge(
        feature_store[['CircuitName', 'CircuitTypeEncoded', 'TotalRaceLaps']],
        on='CircuitName', how='left'
    )

    data['CompoundOffEncoded'] = data['CompoundOff'].map(COMPOUND_MAP)
    data = data.dropna(subset = ['CompoundOffEncoded', 'Stint', 'LapNumber', 'TotalRaceLaps'])

    features = ['Stint', 'CompoundOffEncoded', 'CircuitTypeEncoded', 'TotalRaceLaps']
    X = data[features]
    y = data['LapNumber']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = GradientBoostingRegressor(
        n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)

    log.info(f" Trained on {len(X_train)} pit stops, tested on {len(X_test)}")
    log.info(f" MAE: {mae:.2f} laps")

    return model, {'model': 'PitWindowPredictor', 'MAE_laps': {mae, 2}}

MODEL 5 - CIRCUIT CLUSTERING (K-MEANS)

Groups the 23 circuits by degradation behavior and characterstics - no labels, purely unsupervised

In [9]:
def train_circuit_clustering(feature_store):
    log.info("MODEL 5: CIRCUIT CLUSTERING (K-MEANS)")
    log.info("-" * 50)

    cluster_features = [
        'AvgTrackTemp', 'AvgPitLoss', 'SafetyCarProbability',
        'AvgStintLength', 'MaxObservedTyreLife', 
        'FastestLapSeconds', 'AvgRaceLapSeconds', 'CircuitTypeEncoded'
    ]

    data = feature_store.dropna(subset = cluster_features).copy()
    X = data[cluster_features]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    data['Cluster'] = kmeans.fit_predict(X_scaled)

    log.info(f" Clustered {len(data)} circuits into 4 groups")
    log.info(f"\n{data[['CircuitName', 'Cluster']].sort_values('Cluster').to_string(index = False)}")

    cluster_summary = data.groupby('Cluster')[cluster_features].mean().round(2)
    log.info(f"\nCluster characterstics:\n{cluster_summary.to_string}")

    return kmeans, scaler, data[['CircuitName', 'Cluster']], {'model': 'Circuitclustering', 'NumClusters': 4, 'Circuits': len(data)}

MODEL 6 - UNDERCUT VIABILITY CLASSIFIER

Binary classifier estimating whether a pit stop is likely to result in a position improvement shortly after - documented proxy for undercut success since exact rival gap data wasn't collected in Phase 1

In [10]:
def train_undercut_classifier(pit_stops, raw_laps, feature_store):
    log.info("MODEL 6: UNDERCUT VIABILITY CLASSIFIER")
    log.info("-" * 50)

    rows = []

    for _, stop in pit_stops.iterrows():
        driver = stop['Driver']
        circuit = stop['CircuitName']
        pit_lap = stop['LapNumber']

        before = raw_laps[
            (raw_laps['Driver'] == driver) &
            (raw_laps['CircuitName'] == circuit) &
            (raw_laps['LapNumber'] == pit_lap - 1)
        ]
        after = raw_laps[
            (raw_laps['Driver'] == driver) &
            (raw_laps['CircuitName'] == circuit) &
            (raw_laps['LapNumber'] == pit_lap + 3)
        ]

        if before.empty or after.empty:
            continue
        if pd.isna(before['Position'].values[0]) or pd.isna(after['Position'].values[0]):
            continue

        pos_before = before['Position'].values[0]
        pos_after = after['Position'].values[0]

        #Success proxy: position held or improved 3 laps after the stop
        success = int(pos_after <= pos_before)

        tyre_life_at_pit = before['TyreLife'].values[0] \
            if 'TyreLife' in before.columns else np.nan

        rows.append({
            'CircuitName': circuit,
            'PositionBeforePit': pos_before,
            'TyreLifeAtPit': tyre_life_at_pit,
            'CompoundOffEncoded': COMPOUND_MAP.get(stop['CompoundOff'], np.nan),
            'PitDurationSeconds': stop['PitDurationSeconds'],
            'Success': success
        })

    data = pd.DataFrame(rows).dropna()
    data = data.merge(
        feature_store[['CircuitName', 'CircuitTypeEncoded']],
        on='CircuitName', how='left'
    ).dropna()

    log.info(f" Built proxy database: {len(data)} pit stop events")
    log.info(f" Success rate: {data['Success'].mean():.2f}")

    features = ['PositionBeforePit', 'TyreLifeAtPit', 'CompoundOffEncoded',
                'PitDurationSeconds', 'CircuitTypeEncoded']

    X = data[features]
    y = data['Success']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.fit_transform(X_test)

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train_scaled, y_train)

    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)

    log.info(f" Trained on {len(X_train)} events, tested on {len(X_test)}")
    log.info(f" Accuracy: {acc:.3f}")

    return model, scaler, {'model': 'UndercutClassifier', 'Accuracy': round(acc, 3),
                           'SuccessRate': round(data['Success'].mean(), 3)}

MAIN PIPELINE

In [11]:
def main():
    log.info(" BOXBOX: PHASE 6 MODELS")
    log.info("-" * 50)

    engineered, feature_store, pit_stops, race_results, raw_laps = load_data()

    all_metrics = []

    stint_model, m1 = train_stint_length_predictor(engineered)
    all_metrics.append(m1)

    compound_model, m2 = train_compound_selector(engineered)
    all_metrics.append(m2)

    outcome_model, m3 = train_strategy_outcome_predictor(engineered, race_results, feature_store)
    all_metrics.append(m3)

    pit_window_model, m4 = train_pit_window_predictor(pit_stops, feature_store)
    all_metrics.append(m4)

    kmeans_model, cluster_scaler, cluster_assignments, m5 = train_circuit_clustering(feature_store)
    all_metrics.append(m5)

    undercut_model, undercut_scaler, m6 = train_undercut_classifier(pit_stops, raw_laps, feature_store)
    all_metrics.append(m6)

    #Saving all models
    joblib.dump(stint_model, os.path.join(BASE, 'models', 'stint_length_predictor.pkl'))
    joblib.dump(compound_model, os.path.join(BASE, 'models', 'compound_selector.pkl'))
    joblib.dump(outcome_model, os.path.join(BASE, 'models', 'strategy_outcome.pkl'))
    joblib.dump(pit_window_model, os.path.join(BASE, 'models', 'pit_window_predictor.pkl'))
    joblib.dump(
        {'model': kmeans_model, 'scaler': cluster_scaler},
        os.path.join(BASE, 'models', 'circuit_clusters.pkl')
    )
    joblib.dump(
        {'model': undercut_model, 'scaler': undercut_scaler},
        os.path.join(BASE, 'models', 'undercut_classifier.pkl')
    )

    #Save cluster assignments (used directly by dashboard) 
    cluster_path = os.path.join(BASE, 'data', 'processed', 'circuit_clusters.csv')
    cluster_assignments.to_csv(cluster_path, index=False)
    log.info(f"{len(cluster_path)} Saved")

    #Save metrics summary 
    metrics_df = pd.DataFrame(all_metrics)
    metrics_path = os.path.join(BASE, 'data', 'outputs', 'phase6_model_metrics.csv')
    metrics_df.to_csv(metrics_path, index=False)

    log.info(f"\nPHASE 6 DONE")
    log.info(f"\n{metrics_df.to_string(index=False)}")


if __name__ == '__main__':
    main()

2026-08-01 03:46:46,067 - INFO -  BOXBOX: PHASE 6 MODELS
2026-08-01 03:46:46,068 - INFO - --------------------------------------------------
2026-08-01 03:46:46,069 - INFO - Loading data...
2026-08-01 03:46:46,282 - INFO -  Engineered_laps: 20887 rows
2026-08-01 03:46:46,283 - INFO -  feature_stops: 23 rows
2026-08-01 03:46:46,284 - INFO -  pit_stops_clean: 769 rows
2026-08-01 03:46:46,284 - INFO -  race_results: 479 rows
2026-08-01 03:46:46,285 - INFO -  raw_laps: 26604 rows
2026-08-01 03:46:46,285 - INFO - MODEL-1: STINT LENGTH PREDICTOR
2026-08-01 03:46:46,286 - INFO - --------------------------------------------------
2026-08-01 03:46:46,446 - INFO -  Trained on 860 stints, tested on 215
2026-08-01 03:46:46,446 - INFO -  MAE: 5.48 laps
2026-08-01 03:46:46,447 - INFO - MODEL 2: COMPOUND SELECTOR
2026-08-01 03:46:46,447 - INFO - --------------------------------------------------
2026-08-01 03:46:46,910 - INFO -  Trained on 860 stints, tested on 215
2026-08-01 03:46:46,911 - INFO -  A

In [12]:
import sklearn
print(sklearn.__version__)

1.9.0
